In [1]:
import numpy as np
import pandas as pd
from decimal import Decimal
import re

# Setup and preprocess data

In [3]:
bitcoin_blocks = pd.read_csv("../../data/real_bitcoin_blocks_raw.csv")
coinbase_addresses = pd.read_csv("../../data/coinbase_addresses_full.csv")

In [4]:
# Formatting the real_outputs column data with regex and decimal conversion

# This regex restores the comma between entries before we hand the string to eval().
def parse_outputs(s):
    fixed = re.sub(r"\}\s*\n\s*\{", "}, {", s)
    return eval(fixed, {"Decimal": Decimal, "__builtins__": {}})

coinbase_addresses["real_outputs_parsed"] = coinbase_addresses["real_outputs"].apply(parse_outputs)
coinbase_addresses["n_addresses"] = coinbase_addresses["real_outputs_parsed"].apply(len)

In [5]:
coinbase_addresses['block_timestamp'] = pd.to_datetime(coinbase_addresses['block_timestamp'])
coinbase_addresses['block_timestamp'].describe()

count                              810909
mean     2016-05-13 12:52:27.924965+00:00
min             2009-01-03 18:15:05+00:00
25%             2012-10-10 22:19:35+00:00
50%             2016-04-02 20:55:40+00:00
75%             2019-12-15 06:03:40+00:00
max             2023-10-06 13:37:21+00:00
Name: block_timestamp, dtype: object

In [6]:
def get_primary_miner(coinbase_payouts):
    if coinbase_payouts == []:
        return (None, 0)
    
    highest_payout = max(coinbase_payouts, key=lambda x: x["value"])
    
    highest_payouts = []
    for payout in coinbase_payouts:
        if payout["value"] == highest_payout["value"]:
            highest_payouts.append(payout)
    
    return (highest_payouts[0]["address"], len(highest_payouts))

In [7]:
coinbase_addresses['primary_miner_result'] = coinbase_addresses['real_outputs_parsed'].apply(get_primary_miner)
coinbase_addresses['primary_miner_address'] = coinbase_addresses['primary_miner_result'].str[0]
coinbase_addresses['primary_miner_ties'] = coinbase_addresses['primary_miner_result'].str[1]
# coinbase_addresses

In [8]:
# join both dataframes
merged_df = pd.merge(bitcoin_blocks, coinbase_addresses, on='block_number', how='inner')
merged_df

,number,timestamp,size,transaction_count,bits,block_number,total_output_satoshis,total_output_satoshis_excl_coinbase,total_fee_satoshis,tx_count_check,difficulty,block_timestamp,real_outputs,real_outputs_parsed,n_addresses,primary_miner_result,primary_miner_address,primary_miner_ties
0,0,2009-01-03 18:15:05+00:00,285,1,1d00ffff,0,5.000000e+09,0.000000e+00,0.0,1,1.000000e+00,2009-01-03 18:15:05+00:00,[{'address': '1A1zP1eP5QGefi2DMPTfTL5SLmv7Divf...,[{'address': '1A1zP1eP5QGefi2DMPTfTL5SLmv7Divf...,1,"(1A1zP1eP5QGefi2DMPTfTL5SLmv7DivfNa, 1)",1A1zP1eP5QGefi2DMPTfTL5SLmv7DivfNa,1
1,1,2009-01-09 02:54:25+00:00,215,1,1d00ffff,1,5.000000e+09,0.000000e+00,0.0,1,1.000000e+00,2009-01-09 02:54:25+00:00,[{'address': '12c6DSiU4Rq3P4ZxziKxzrL5LmMBrzjr...,[{'address': '12c6DSiU4Rq3P4ZxziKxzrL5LmMBrzjr...,1,"(12c6DSiU4Rq3P4ZxziKxzrL5LmMBrzjrJX, 1)",12c6DSiU4Rq3P4ZxziKxzrL5LmMBrzjrJX,1
2,2,2009-01-09 02:55:44+00:00,215,1,1d00ffff,2,5.000000e+09,0.000000e+00,0.0,1,1.000000e+00,2009-01-09 02:55:44+00:00,[{'address': '1HLoD9E4SDFFPDiYfNYnkBLQ85Y51J3Z...,[{'address': '1HLoD9E4SDFFPDiYfNYnkBLQ85Y51J3Z...,1,"(1HLoD9E4SDFFPDiYfNYnkBLQ85Y51J3Zb1, 1)",1HLoD9E4SDFFPDiYfNYnkBLQ85Y51J3Zb1,1
3,3,2009-01-09 03:02:53+00:00,215,1,1d00ffff,3,5.000000e+09,0.000000e+00,0.0,1,1.000000e+00,2009-01-09 03:02:53+00:00,[{'address': '1FvzCLoTPGANNjWoUo6jUGuAG3wg1w4Y...,[{'address': '1FvzCLoTPGANNjWoUo6jUGuAG3wg1w4Y...,1,"(1FvzCLoTPGANNjWoUo6jUGuAG3wg1w4YjR, 1)",1FvzCLoTPGANNjWoUo6jUGuAG3wg1w4YjR,1
4,4,2009-01-09 03:16:28+00:00,215,1,1d00ffff,4,5.000000e+09,0.000000e+00,0.0,1,1.000000e+00,2009-01-09 03:16:28+00:00,[{'address': '15ubicBBWFnvoZLT7GiU2qxjRaKJPdkD...,[{'address': '15ubicBBWFnvoZLT7GiU2qxjRaKJPdkD...,1,"(15ubicBBWFnvoZLT7GiU2qxjRaKJPdkDMG, 1)",15ubicBBWFnvoZLT7GiU2qxjRaKJPdkDMG,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
810904,810904,2023-10-06 12:36:45+00:00,1471359,2641,1704e90f,810904,7.607876e+11,7.601375e+11,25173010.0,2641,5.732151e+13,2023-10-06 12:36:45+00:00,[{'address': 'bc1qxhmdufsvnuaaaer4ynz88fspdsxq...,[{'address': 'bc1qxhmdufsvnuaaaer4ynz88fspdsxq...,1,"(bc1qxhmdufsvnuaaaer4ynz88fspdsxq2h9e9cetdj, 1)",bc1qxhmdufsvnuaaaer4ynz88fspdsxq2h9e9cetdj,1
810905,810905,2023-10-06 12:45:49+00:00,1427754,2288,1704e90f,810905,1.161340e+12,1.160692e+12,23663392.0,2288,5.732151e+13,2023-10-06 12:45:49+00:00,[{'address': 'bc1qxhmdufsvnuaaaer4ynz88fspdsxq...,[{'address': 'bc1qxhmdufsvnuaaaer4ynz88fspdsxq...,1,"(bc1qxhmdufsvnuaaaer4ynz88fspdsxq2h9e9cetdj, 1)",bc1qxhmdufsvnuaaaer4ynz88fspdsxq2h9e9cetdj,1
810906,810906,2023-10-06 12:50:15+00:00,1612966,2016,1704e90f,810906,4.643628e+11,4.637224e+11,15483511.0,2016,5.732151e+13,2023-10-06 12:50:15+00:00,[{'address': '1K6KoYC69NnafWJ7YgtrpwJxBLiijWqw...,[{'address': '1K6KoYC69NnafWJ7YgtrpwJxBLiijWqw...,2,"(1KFHE7w8BhaENAswwryaoccDb6qcT6DbYY, 1)",1KFHE7w8BhaENAswwryaoccDb6qcT6DbYY,1
810907,810907,2023-10-06 13:37:00+00:00,1486563,3496,1704e90f,810907,2.469789e+12,2.469101e+12,63223285.0,3496,5.732151e+13,2023-10-06 13:37:00+00:00,[{'address': '38XnPvu9PmonFU9WouPXUjYbW91wa5Me...,[{'address': '38XnPvu9PmonFU9WouPXUjYbW91wa5Me...,1,"(38XnPvu9PmonFU9WouPXUjYbW91wa5MerL, 1)",38XnPvu9PmonFU9WouPXUjYbW91wa5MerL,1


In [9]:
miner_block_counts = merged_df['primary_miner_address'].value_counts()
print(miner_block_counts)

primary_miner_address
1KFHE7w8BhaENAswwryaoccDb6qcT6DbYY            81237
18cBEMRxXHqzWWCxZNtU91F5sbUNKhL5PX            34093
1CK6KHY6MHgYvmRQ4PAafKYDrg1ejbH1cE            27261
14cZMQk89mRYQkDEj8Rn25AnGoBi5H6uer            26204
1CjPR7Z5ZSyWk6WtXvSFgkptmpoi4UM9BC            23083
                                              ...  
3EHpvbs5ar7DWiux1AQ5JMB2zTBE6wzX7d                1
bc1qrm09asqsxh2l7pjxjlltm53tx689dp2amzupzq        1
bc1q6fu02uvz2ghz3e3avqvqsqscr2pp4xhnuqfxpa        1
bc1q2za4ejga366sn288273pty8trasn5zs4y9hqg6        1
15h4MFgMs3yGiGfXA7fqhpsTWMkQ95EFBB                1
Name: count, Length: 197095, dtype: int64


In [10]:
# Step A: for every row, look up how many blocks its own address has mined in total
# .map() works like a dictionary lookup applied to an entire column at once,
# and importantly, unlike direct indexing, .map() quietly returns NaN for addresses
# it can't find (like our two None addresses), instead of crashing
merged_df['own_block_count'] = merged_df['primary_miner_address'].map(miner_block_counts)

# Step B: for every row, decide: keep the real address if count >= 10, otherwise 'long_tail'
merged_df['miner_group'] = np.where(merged_df['own_block_count'] >= 10, merged_df['primary_miner_address'], 'long_tail')

merged_df

,number,timestamp,size,transaction_count,bits,block_number,total_output_satoshis,total_output_satoshis_excl_coinbase,total_fee_satoshis,tx_count_check,difficulty,block_timestamp,real_outputs,real_outputs_parsed,n_addresses,primary_miner_result,primary_miner_address,primary_miner_ties,own_block_count,miner_group
0,0,2009-01-03 18:15:05+00:00,285,1,1d00ffff,0,5.000000e+09,0.000000e+00,0.0,1,1.000000e+00,2009-01-03 18:15:05+00:00,[{'address': '1A1zP1eP5QGefi2DMPTfTL5SLmv7Divf...,[{'address': '1A1zP1eP5QGefi2DMPTfTL5SLmv7Divf...,1,"(1A1zP1eP5QGefi2DMPTfTL5SLmv7DivfNa, 1)",1A1zP1eP5QGefi2DMPTfTL5SLmv7DivfNa,1,1.0,long_tail
1,1,2009-01-09 02:54:25+00:00,215,1,1d00ffff,1,5.000000e+09,0.000000e+00,0.0,1,1.000000e+00,2009-01-09 02:54:25+00:00,[{'address': '12c6DSiU4Rq3P4ZxziKxzrL5LmMBrzjr...,[{'address': '12c6DSiU4Rq3P4ZxziKxzrL5LmMBrzjr...,1,"(12c6DSiU4Rq3P4ZxziKxzrL5LmMBrzjrJX, 1)",12c6DSiU4Rq3P4ZxziKxzrL5LmMBrzjrJX,1,1.0,long_tail
2,2,2009-01-09 02:55:44+00:00,215,1,1d00ffff,2,5.000000e+09,0.000000e+00,0.0,1,1.000000e+00,2009-01-09 02:55:44+00:00,[{'address': '1HLoD9E4SDFFPDiYfNYnkBLQ85Y51J3Z...,[{'address': '1HLoD9E4SDFFPDiYfNYnkBLQ85Y51J3Z...,1,"(1HLoD9E4SDFFPDiYfNYnkBLQ85Y51J3Zb1, 1)",1HLoD9E4SDFFPDiYfNYnkBLQ85Y51J3Zb1,1,1.0,long_tail
3,3,2009-01-09 03:02:53+00:00,215,1,1d00ffff,3,5.000000e+09,0.000000e+00,0.0,1,1.000000e+00,2009-01-09 03:02:53+00:00,[{'address': '1FvzCLoTPGANNjWoUo6jUGuAG3wg1w4Y...,[{'address': '1FvzCLoTPGANNjWoUo6jUGuAG3wg1w4Y...,1,"(1FvzCLoTPGANNjWoUo6jUGuAG3wg1w4YjR, 1)",1FvzCLoTPGANNjWoUo6jUGuAG3wg1w4YjR,1,1.0,long_tail
4,4,2009-01-09 03:16:28+00:00,215,1,1d00ffff,4,5.000000e+09,0.000000e+00,0.0,1,1.000000e+00,2009-01-09 03:16:28+00:00,[{'address': '15ubicBBWFnvoZLT7GiU2qxjRaKJPdkD...,[{'address': '15ubicBBWFnvoZLT7GiU2qxjRaKJPdkD...,1,"(15ubicBBWFnvoZLT7GiU2qxjRaKJPdkDMG, 1)",15ubicBBWFnvoZLT7GiU2qxjRaKJPdkDMG,1,1.0,long_tail
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
810904,810904,2023-10-06 12:36:45+00:00,1471359,2641,1704e90f,810904,7.607876e+11,7.601375e+11,25173010.0,2641,5.732151e+13,2023-10-06 12:36:45+00:00,[{'address': 'bc1qxhmdufsvnuaaaer4ynz88fspdsxq...,[{'address': 'bc1qxhmdufsvnuaaaer4ynz88fspdsxq...,1,"(bc1qxhmdufsvnuaaaer4ynz88fspdsxq2h9e9cetdj, 1)",bc1qxhmdufsvnuaaaer4ynz88fspdsxq2h9e9cetdj,1,14480.0,bc1qxhmdufsvnuaaaer4ynz88fspdsxq2h9e9cetdj
810905,810905,2023-10-06 12:45:49+00:00,1427754,2288,1704e90f,810905,1.161340e+12,1.160692e+12,23663392.0,2288,5.732151e+13,2023-10-06 12:45:49+00:00,[{'address': 'bc1qxhmdufsvnuaaaer4ynz88fspdsxq...,[{'address': 'bc1qxhmdufsvnuaaaer4ynz88fspdsxq...,1,"(bc1qxhmdufsvnuaaaer4ynz88fspdsxq2h9e9cetdj, 1)",bc1qxhmdufsvnuaaaer4ynz88fspdsxq2h9e9cetdj,1,14480.0,bc1qxhmdufsvnuaaaer4ynz88fspdsxq2h9e9cetdj
810906,810906,2023-10-06 12:50:15+00:00,1612966,2016,1704e90f,810906,4.643628e+11,4.637224e+11,15483511.0,2016,5.732151e+13,2023-10-06 12:50:15+00:00,[{'address': '1K6KoYC69NnafWJ7YgtrpwJxBLiijWqw...,[{'address': '1K6KoYC69NnafWJ7YgtrpwJxBLiijWqw...,2,"(1KFHE7w8BhaENAswwryaoccDb6qcT6DbYY, 1)",1KFHE7w8BhaENAswwryaoccDb6qcT6DbYY,1,81237.0,1KFHE7w8BhaENAswwryaoccDb6qcT6DbYY
810907,810907,2023-10-06 13:37:00+00:00,1486563,3496,1704e90f,810907,2.469789e+12,2.469101e+12,63223285.0,3496,5.732151e+13,2023-10-06 13:37:00+00:00,[{'address': '38XnPvu9PmonFU9WouPXUjYbW91wa5Me...,[{'address': '38XnPvu9PmonFU9WouPXUjYbW91wa5Me...,1,"(38XnPvu9PmonFU9WouPXUjYbW91wa5MerL, 1)",38XnPvu9PmonFU9WouPXUjYbW91wa5MerL,1,15630.0,38XnPvu9PmonFU9WouPXUjYbW91wa5MerL


In [11]:
# note: only 1 block had 0 in total_output_satoshis
# fill all 0 values with the median total_output_satoshis 
merged_df['total_output_satoshis'] = merged_df['total_output_satoshis'].replace(0, np.nan)
merged_df['total_output_satoshis'] = merged_df['total_output_satoshis'].fillna(merged_df['total_output_satoshis'].median())

# Testing

In [12]:
from scipy.stats import spearmanr

def split_halves_generic(data, min_blocks, value_col, group_col='miner_group'):
    """Per-miner first-half vs second-half mean of `value_col`."""
    df = data[data[group_col] != 'long_tail'].copy()
    counts = df[group_col].value_counts()
    df = df[df[group_col].isin(counts[counts >= min_blocks].index)].copy()

    df = df.sort_values([group_col, 'block_number'])
    df['pos']      = df.groupby(group_col).cumcount()
    df['n_blocks'] = df.groupby(group_col)['block_number'].transform('size')
    df['half']     = np.where(df['pos'] < df['n_blocks'] // 2, 'first', 'second')

    return df.groupby([group_col, 'half'])[value_col].mean().unstack().dropna()

In [13]:
# How much of the chain has usable fee data?
print("zero-fee blocks:", (merged_df['total_fee_satoshis'] == 0).sum())
print("as fraction:", (merged_df['total_fee_satoshis'] == 0).mean().round(4))

# Where are they? (expecting early chain)
zero_fee = merged_df[merged_df['total_fee_satoshis'] == 0]
print(zero_fee['block_number'].describe())

# Fee scale over time — confirms era correction is needed
merged_df['year'] = pd.to_datetime(merged_df['timestamp']).dt.year
print(merged_df.groupby('year')['total_fee_satoshis'].median())

zero-fee blocks: 125807
as fraction: 0.1551
count    125807.000000
mean      80139.295429
std       92126.178628
min           0.000000
25%       31459.500000
50%       62917.000000
75%       95071.500000
max      810811.000000
Name: block_number, dtype: float64
year
2009            0.0
2010            0.0
2011      1200000.0
2012      4300009.5
2013     13265704.0
2014      5747890.0
2015     11220998.0
2016     36431702.0
2017    146467309.5
2018     15395445.0
2019     22981160.0
2020     31677821.0
2021     18655396.5
2022      7870688.5
2023     16963314.0
Name: total_fee_satoshis, dtype: float64


In [14]:
# What does a 2011 cutoff cost us?
merged_df['timestamp'] = pd.to_datetime(merged_df['timestamp'])
CUTOFF = pd.Timestamp('2011-01-01', tz='UTC')

before = merged_df[merged_df['timestamp'] < CUTOFF]
after  = merged_df[merged_df['timestamp'] >= CUTOFF]

print(f"blocks dropped: {len(before):,} ({len(before)/len(merged_df):.2%})")
print(f"blocks kept:    {len(after):,}")

# The number that actually matters: trackable miners at each threshold,
# recomputed on the post-cutoff data only
for min_blocks in [20, 30, 50, 100]:
    counts = (after[after['miner_group'] != 'long_tail']['miner_group']
                .value_counts())
    n_full = (merged_df[merged_df['miner_group'] != 'long_tail']['miner_group']
                .value_counts() >= min_blocks).sum()
    print(f"min_blocks={min_blocks:3d} | full chain: {n_full:4d} miners"
          f" | post-2011: {(counts >= min_blocks).sum():4d} miners")

# Zero-fee blocks remaining after cutoff (these are real empty blocks, kept deliberately)
print(f"\nzero-fee blocks post-cutoff: {(after['total_fee_satoshis'] == 0).sum():,}"
      f" ({(after['total_fee_satoshis'] == 0).mean():.2%})")

blocks dropped: 100,410 (12.38%)
blocks kept:    710,499
min_blocks= 20 | full chain:  647 miners | post-2011:  647 miners
min_blocks= 30 | full chain:  537 miners | post-2011:  537 miners
min_blocks= 50 | full chain:  430 miners | post-2011:  430 miners
min_blocks=100 | full chain:  317 miners | post-2011:  317 miners

zero-fee blocks post-cutoff: 26,319 (3.70%)


In [15]:
# Sanity: do the >=20-block miners have any pre-2011 presence at all?
tracked = merged_df[merged_df['miner_group'] != 'long_tail']
counts_full = tracked['miner_group'].value_counts()
big = counts_full[counts_full >= 20].index

pre = merged_df[(merged_df['timestamp'] < CUTOFF) & (merged_df['miner_group'].isin(big))]
print("pre-2011 blocks owned by >=20-block miners:", len(pre))

pre_all = merged_df[merged_df['timestamp'] < CUTOFF]
print("pre-2011 blocks in long_tail:", (pre_all['miner_group'] == 'long_tail').sum(),
      "of", len(pre_all))

pre-2011 blocks owned by >=20-block miners: 0
pre-2011 blocks in long_tail: 100410 of 100410


In [17]:
pre_all = merged_df[merged_df['timestamp'] < CUTOFF]
print("distinct addresses pre-2011:", pre_all['primary_miner_address'].nunique())
print("max blocks by any one pre-2011 address:",
      pre_all['primary_miner_address'].value_counts().max())

distinct addresses pre-2011: 100404
max blocks by any one pre-2011 address: 2


In [16]:
# --- Approach E: fee-capture label, era-corrected persistence ---
# Mirrors 84.4's design so results sit alongside the transactions-per-satoshi ones.

work_e = merged_df[merged_df['timestamp'] >= CUTOFF].copy()

# Two candidate outcome definitions (see note below on why both)
work_e['fee_raw']     = work_e['total_fee_satoshis']
work_e['fee_density'] = work_e['total_fee_satoshis'] / work_e['size']

work_e['year']  = work_e['timestamp'].dt.to_period('Y')
work_e['month'] = work_e['timestamp'].dt.to_period('M')

# Percentile-rank within era, against ALL blocks including long_tail —
# the reference population is everything mined at that time, not just tracked miners.
for base in ['fee_raw', 'fee_density']:
    work_e[f'{base}_pct_year']  = work_e.groupby('year')[base].rank(pct=True)
    work_e[f'{base}_pct_month'] = work_e.groupby('month')[base].rank(pct=True)


def persistence_row(data, min_blocks, value_col):
    halves = split_halves_generic(data, min_blocks, value_col)
    rho, _ = spearmanr(halves['first'], halves['second'])
    lab_f = (halves['first']  > halves['first'].median()).astype(int)
    lab_s = (halves['second'] > halves['second'].median()).astype(int)
    return len(halves), rho, (lab_f == lab_s).mean()


rows = []
for base in ['fee_raw', 'fee_density']:
    for min_blocks in [20, 30, 50, 100]:
        row = {'label': base, 'min_blocks': min_blocks}
        for tag, col in [('raw',   base),
                         ('year',  f'{base}_pct_year'),
                         ('month', f'{base}_pct_month')]:
            n, rho, agree = persistence_row(work_e, min_blocks, col)
            row['n_miners']     = n
            row[f'rho_{tag}']   = rho
            row[f'agree_{tag}'] = agree
        rows.append(row)

fee_persistence = pd.DataFrame(rows)[
    ['label', 'min_blocks', 'n_miners',
     'rho_raw', 'rho_year', 'rho_month',
     'agree_raw', 'agree_year', 'agree_month']
]
fee_persistence.round(4)

C:\Users\KwokYenMing\AppData\Local\Temp\ipykernel_27200\3426602086.py:10: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  work_e['year']  = work_e['timestamp'].dt.to_period('Y')
C:\Users\KwokYenMing\AppData\Local\Temp\ipykernel_27200\3426602086.py:11: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  work_e['month'] = work_e['timestamp'].dt.to_period('M')


,label,min_blocks,n_miners,rho_raw,rho_year,rho_month,agree_raw,agree_year,agree_month
0,fee_raw,20,647,0.7944,0.4744,0.3758,0.8083,0.6352,0.6291
1,fee_raw,30,537,0.7863,0.4485,0.3743,0.7877,0.6164,0.6164
2,fee_raw,50,430,0.7651,0.4025,0.3783,0.7674,0.6000,0.6093
3,fee_raw,100,317,0.7470,0.3237,0.3892,0.7603,0.5710,0.6341
4,fee_density,20,647,0.8213,0.5736,0.4103,0.8578,0.7063,0.6291
5,fee_density,30,537,0.8094,0.5613,0.4324,0.8399,0.6909,0.6387
6,fee_density,50,430,0.7869,0.5144,0.4900,0.8186,0.6605,0.6512
7,fee_density,100,317,0.7790,0.4154,0.4347,0.7981,0.6215,0.6215


In [18]:
work_e['week'] = work_e['timestamp'].dt.to_period('W')
for base in ['fee_raw', 'fee_density']:
    work_e[f'{base}_pct_week'] = work_e.groupby('week')[base].rank(pct=True)

rows = []
for base in ['fee_raw', 'fee_density']:
    for min_blocks in [20, 30, 50, 100]:
        n, rho, agree = persistence_row(work_e, min_blocks, f'{base}_pct_week')
        rows.append({'label': base, 'min_blocks': min_blocks,
                     'n_miners': n, 'rho_week': rho, 'agree_week': agree})

pd.DataFrame(rows).round(4)

C:\Users\KwokYenMing\AppData\Local\Temp\ipykernel_27200\1169475956.py:1: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  work_e['week'] = work_e['timestamp'].dt.to_period('W')


,label,min_blocks,n_miners,rho_week,agree_week
0,fee_raw,20,647,0.3924,0.6352
1,fee_raw,30,537,0.4036,0.6425
2,fee_raw,50,430,0.4334,0.6372
3,fee_raw,100,317,0.4721,0.6593
4,fee_density,20,647,0.4574,0.6538
5,fee_density,30,537,0.4862,0.6574
6,fee_density,50,430,0.5781,0.6791
7,fee_density,100,317,0.5455,0.6467


In [21]:
# --- Permutation null: what does this procedure score on data with no real signal? ---
# Shuffles miner labels within each month: preserves each month's active miner set
# and their block counts, destroys which specific blocks each one produced.

rng = np.random.default_rng(42)
null_rows = []

for rep in range(20):
    shuf = work_e.copy()
    shuf['miner_group'] = (
        shuf.groupby('month')['miner_group']
            .transform(lambda s: rng.permutation(s.to_numpy()))
    )
    for min_blocks in [20, 50]:
        for tag, col in [('year',  'fee_density_pct_year'),
                         ('month', 'fee_density_pct_month'),
                         ('week',  'fee_density_pct_week')]:
            n, rho, agree = persistence_row(shuf, min_blocks, col)
            null_rows.append({'rep': rep, 'min_blocks': min_blocks,
                              'window': tag, 'n_miners': n,
                              'rho': rho, 'agree': agree})

null_df = pd.DataFrame(null_rows)
print(null_df.groupby(['min_blocks', 'window'])[['rho', 'agree']]
             .agg(['mean', 'std']).round(4))

                      rho           agree        
                     mean     std    mean     std
min_blocks window                                
20         month  -0.1150  0.0392  0.4578  0.0187
           week   -0.0172  0.0338  0.4947  0.0147
           year    0.4321  0.0149  0.6410  0.0105
50         month  -0.1537  0.0574  0.4472  0.0272
           week   -0.0105  0.0494  0.4974  0.0209
           year    0.3945  0.0179  0.6216  0.0152


In [20]:
# --- Null for the ORIGINAL efficiency label (the 84.4 procedure) ---
# Same shuffle design. Uses the full chain, not the 2011 cutoff, to match 84.4 exactly.

null_eff = merged_df.copy()
null_eff['timestamp']        = pd.to_datetime(null_eff['timestamp'])
null_eff['block_efficiency'] = null_eff['transaction_count'] / null_eff['total_output_satoshis']
null_eff['year']  = null_eff['timestamp'].dt.to_period('Y')
null_eff['month'] = null_eff['timestamp'].dt.to_period('M')
null_eff['eff_pct_year']  = null_eff.groupby('year')['block_efficiency'].rank(pct=True)
null_eff['eff_pct_month'] = null_eff.groupby('month')['block_efficiency'].rank(pct=True)

rng = np.random.default_rng(42)
rows = []
for rep in range(20):                      # 20, not 5 — this corrects a report number
    shuf = null_eff.copy()
    shuf['miner_group'] = (shuf.groupby('month')['miner_group']
                               .transform(lambda s: rng.permutation(s.to_numpy())))
    for min_blocks in [20, 30, 50, 100]:
        for tag, col in [('year', 'eff_pct_year'), ('month', 'eff_pct_month')]:
            n, rho, agree = persistence_row(shuf, min_blocks, col)
            rows.append({'rep': rep, 'min_blocks': min_blocks, 'window': tag,
                         'rho': rho, 'agree': agree})

eff_null = pd.DataFrame(rows)
print(eff_null.groupby(['min_blocks', 'window'])[['rho', 'agree']]
              .agg(['mean', 'std']).round(4))

C:\Users\KwokYenMing\AppData\Local\Temp\ipykernel_27200\3275382526.py:7: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  null_eff['year']  = null_eff['timestamp'].dt.to_period('Y')
C:\Users\KwokYenMing\AppData\Local\Temp\ipykernel_27200\3275382526.py:8: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  null_eff['month'] = null_eff['timestamp'].dt.to_period('M')


                      rho           agree        
                     mean     std    mean     std
min_blocks window                                
20         month  -0.0625  0.0447  0.4705  0.0209
           year    0.1551  0.0220  0.5332  0.0125
30         month  -0.0866  0.0508  0.4646  0.0232
           year    0.1523  0.0182  0.5266  0.0142
50         month  -0.0802  0.0682  0.4656  0.0289
           year    0.1337  0.0191  0.5188  0.0166
100        month  -0.0930  0.0658  0.4618  0.0292
           year    0.1231  0.0230  0.5066  0.0194
